# Iris: классификация трёх видов цветка

В этом примере сеть классифицирует цветки ириса по четырём измерениям: длине и ширине чашелистика и лепестка. Набор содержит три вида цветка. Мы разделим его на обучающую и тестовую части, обучим многослойную сеть и оценим её на отложенных примерах.

Покажем также подготовку признаков: масштабирование по статистикам только обучающей выборки и one-hot кодирование классов для categorical cross-entropy.


In [635]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from open_neuron.activations import softmax, tanh
from open_neuron.layers import Layer
from open_neuron.models import NeuralNetwork
from open_neuron.optimizers import Adam

np.set_printoptions(precision=3, suppress=True)
iris = load_iris()
X = iris.data
y = iris.target
class_names = iris.target_names

print("Признаки:", iris.feature_names)
print("Классы:", class_names)
print("Размер данных:", X.shape)

Признаки: ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']
Классы: ['setosa' 'versicolor' 'virginica']
Размер данных: (150, 4)


## Делим данные и масштабируем признаки

`stratify=y` сохраняет доли трёх видов в обеих выборках, а `random_state` делает разбиение повторяемым. Масштабировщик обучается только на `X_train`; к тестовым данным применяется уже найденное преобразование. Так сведения из тестовой выборки не участвуют в подготовке обучения.

Метки классов хранятся целыми числами. Для сети преобразуем только обучающие метки в one-hot матрицу: например, класс `2` станет `[0, 0, 1]`.


In [636]:
X_train, X_test, y_train_labels, y_test_labels = train_test_split(
    X,
    y,
    test_size=0.2,
    # random_state=42,
    stratify=y,
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
y_train = np.eye(len(class_names))[y_train_labels]

print("Обучающих примеров:", len(X_train))
print("Тестовых примеров:  ", len(X_test))
print("Средние обучающих признаков после масштабирования:",
      X_train_scaled.mean(axis=0))
print("One-hot метки первых трёх примеров:")
print(y_train[:3])

Обучающих примеров: 120
Тестовых примеров:   30
Средние обучающих признаков после масштабирования: [ 0. -0. -0. -0.]
One-hot метки первых трёх примеров:
[[0. 1. 0.]
 [0. 1. 0.]
 [0. 1. 0.]]


## Строим и обучаем сеть

Скрытый слой содержит четыре нейрона с `tanh`. Выходной слой содержит по одному нейрону на каждый вид ириса; `softmax` превращает их значения в распределение вероятностей. Категориальная кросс-энтропия сравнивает это распределение с one-hot ответами.

Во время обучения сеть видит только масштабированные обучающие примеры. Тестовую выборку не передаём в `fit`; используем её один раз после обучения для оценки обобщения.


In [637]:
# np.random.seed(7)
model = NeuralNetwork(
    [
        Layer(4, activation=tanh),
        Layer(len(class_names), activation=softmax),
    ],
    inputs_dim=X_train_scaled.shape[1],
    loss="categorical_crossentropy",
    optimizer=Adam(learning_rate=0.01),
)

model.fit(
    X_train_scaled,
    y_train,
    epochs=10,
    # batch_size=2,
    shuffle=True,
)

Training started with Overall Loss 0.7435 on Train Data
Epoch 1/10, Loss: 0.4094, Validation Loss: 0.2551 on Train Data
Epoch 2/10, Loss: 0.2117, Validation Loss: 0.1569 on Train Data
Epoch 3/10, Loss: 0.1448, Validation Loss: 0.1186 on Train Data
Epoch 4/10, Loss: 0.1106, Validation Loss: 0.0880 on Train Data
Epoch 5/10, Loss: 0.0932, Validation Loss: 0.0815 on Train Data
Epoch 6/10, Loss: 0.0891, Validation Loss: 0.0688 on Train Data
Epoch 7/10, Loss: 0.0743, Validation Loss: 0.0614 on Train Data
Epoch 8/10, Loss: 0.0707, Validation Loss: 0.1066 on Train Data
Epoch 9/10, Loss: 0.0741, Validation Loss: 0.0561 on Train Data
Epoch 10/10, Loss: 0.0679, Validation Loss: 0.0555 on Train Data
Training completed with Overall Loss 0.0555


## Оцениваем на отложенной выборке

`argmax` выбирает вид с наибольшей предсказанной вероятностью. Матрица ошибок показывает, какие виды сеть путает; отчёт отдельно показывает precision, recall и F1 для каждого вида. Результат относится к конкретному фиксированному разбиению, а не гарантирует такую же точность на любом наборе данных.


In [638]:
probabilities = model.predict(X_test_scaled)
predicted_labels = np.argmax(probabilities, axis=1)

print(
    f"Точность на тестовой выборке: {accuracy_score(y_test_labels, predicted_labels):.1%}")
print("Матрица ошибок (строки — ответ, столбцы — предсказание):")
print(confusion_matrix(y_test_labels, predicted_labels,
      labels=np.arange(len(class_names))))
print("Отчёт по классам:")
print(classification_report(
    y_test_labels,
    predicted_labels,
    labels=np.arange(len(class_names)),
    target_names=class_names,
    zero_division=0,
))

print("Первые предсказания:")
for expected, predicted, probabilities_for_sample in zip(
    y_test_labels[:5], predicted_labels[:5], probabilities[:5]
):
  print(
      f"ответ: {class_names[expected]}, "
      f"предсказание: {class_names[predicted]}, "
      f"вероятности: {probabilities_for_sample}"
  )

Точность на тестовой выборке: 93.3%
Матрица ошибок (строки — ответ, столбцы — предсказание):
[[10  0  0]
 [ 0  9  1]
 [ 0  1  9]]
Отчёт по классам:
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       0.90      0.90      0.90        10
   virginica       0.90      0.90      0.90        10

    accuracy                           0.93        30
   macro avg       0.93      0.93      0.93        30
weighted avg       0.93      0.93      0.93        30

Первые предсказания:
ответ: virginica, предсказание: versicolor, вероятности: [0.001 0.504 0.495]
ответ: virginica, предсказание: virginica, вероятности: [0.    0.301 0.698]
ответ: versicolor, предсказание: versicolor, вероятности: [0.001 0.985 0.013]
ответ: virginica, предсказание: virginica, вероятности: [0.   0.01 0.99]
ответ: virginica, предсказание: virginica, вероятности: [0.    0.007 0.993]


In [639]:
model.summary()

Info:
Network type: "Classification", layers: 2, neurons: 7, optimizer: "Adam", epochs: 10, batch size: 1, loss function: "Categorical Crossentropy"
Last forward:
Layer 1, neurons: 4, inputs: [-0.302 -1.219  0.09  -0.11 ], activation: tanh, output (a): [0.946 0.802 0.78  0.931]
Unit 1/4, inputs: [-0.302 -1.219  0.09  -0.11 ], weights: [ 0.386 -0.938  1.156  1.564], bias: [0.835], z: 1.794164938393466, activation: tanh, a: 0.9461983933518809
Unit 2/4, inputs: [-0.302 -1.219  0.09  -0.11 ], weights: [-0.02   0.381 -1.274 -1.045], bias: [1.563], z: 1.1047391382921352, activation: tanh, a: 0.8021948802397662
Unit 3/4, inputs: [-0.302 -1.219  0.09  -0.11 ], weights: [ 0.417 -0.834 -0.593 -0.986], bias: [0.099], z: 1.0448147860587886, activation: tanh, a: 0.7797822690782107
Unit 4/4, inputs: [-0.302 -1.219  0.09  -0.11 ], weights: [ 0.228 -0.489  1.27   1.013], bias: [1.135], z: 1.666143933838479, activation: tanh, a: 0.9310400331022153
Layer 2, neurons: 3, inputs: [0.946 0.802 0.78  0.931],